# Chatbot para Reservas de Hotel
Este cuaderno ejecuta un chatbot sencillo con procesamiento de lenguaje natural usando spaCy.
Puedes reservar, consultar precios o cancelar reservas de manera simulada.

In [ ]:
!pip install -q spacy
!python -m spacy download es_core_news_sm

## Ejercicio 1: Intención sobre servicios del hotel
Se añade la intención servicios para responder consultas sobre piscina, desayuno, wifi, parking, gimnasio y spa.

In [2]:
SERVICIOS_HOTEL = {
    "piscina": "Sí, disponemos de piscina exterior climatizada abierta de 09:00 a 22:00.",
    "desayuno": "El desayuno buffet está incluido en todas las reservas, servido de 07:00 a 10:30.",
    "wifi": "Ofrecemos WiFi gratuito de alta velocidad en todas las habitaciones y zonas comunes.",
    "parking": "Disponemos de parking privado con un coste adicional de 10€/día.",
    "gimnasio": "El gimnasio está disponible 24 horas para los huéspedes.",
    "spa": "Contamos con spa y sauna. Reserva previa en recepción."
}

def responder_servicios(texto):
    texto = texto.lower()
    respuestas = [info for clave, info in SERVICIOS_HOTEL.items() if clave in texto]
    if respuestas:
        return "\n".join(respuestas)
    return ("Nuestros servicios incluyen: "
            + ", ".join(SERVICIOS_HOTEL.keys())
            + ". ¿Sobre cuál desea más información?")

## Ejercicio 2: Tipo de habitación con precios diferenciados
Se añaden tres tipos de habitación (individual, doble, suite) con precios distintos. La consulta y la reserva los tienen en cuenta.

In [3]:
TIPOS_HABITACION = {
    "individual": 55,
    "doble": 80,
    "suite": 150
}

def extraer_tipo_habitacion(texto):
    texto = texto.lower()
    for tipo in TIPOS_HABITACION:
        if tipo in texto:
            return tipo
    return None

def mostrar_precios():
    lineas = [f"- {tipo.capitalize()}: {precio}€/noche" for tipo, precio in TIPOS_HABITACION.items()]
    return "Nuestras tarifas son:\n" + "\n".join(lineas)

## Ejercicio 3: Persistencia en CSV
Las reservas se cargan desde reservas.csv al iniciar y se escriben tras cada alta o cancelación.

In [4]:
import csv
import os

CSV_PATH = "reservas.csv"
CAMPOS_CSV = ["id", "entrada", "salida", "personas", "tipo", "total"]

def cargar_reservas(reservas):
    siguiente_id = 1
    if not os.path.exists(CSV_PATH):
        return siguiente_id
    with open(CSV_PATH, "r", newline="", encoding="utf-8") as f:
        lector = csv.DictReader(f)
        for fila in lector:
            fila["id"] = int(fila["id"])
            fila["personas"] = int(fila["personas"])
            fila["total"] = float(fila["total"]) if fila.get("total") else 0
            reservas.append(fila)
    if reservas:
        siguiente_id = max(r["id"] for r in reservas) + 1
    return siguiente_id

def guardar_reservas(reservas):
    with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
        escritor = csv.DictWriter(f, fieldnames=CAMPOS_CSV)
        escritor.writeheader()
        escritor.writerows(reservas)

## Ejercicio 4: Control de errores en formato de fecha
Se detectan formatos incorrectos habituales (AAAA/MM/DD, DD-MM-AAAA, etc.) y se reintenta la validación hasta obtener fechas válidas.

In [5]:
FORMATOS_ADMITIDOS = ['%d/%m/%Y', '%d-%m-%Y', '%Y/%m/%d', '%Y-%m-%d', '%d.%m.%Y']

def extraer_fechas_flexible(texto):
    patron = r'\d{1,4}[/\-.]\d{1,2}[/\-.]\d{1,4}'
    return re.findall(patron, texto)

def parsear_fecha(fecha_str):
    for fmt in FORMATOS_ADMITIDOS:
        try:
            fecha = datetime.strptime(fecha_str, fmt)
            if fmt != '%d/%m/%Y':
                print(f"Bot Hotel: Aviso - la fecha '{fecha_str}' no está en el formato recomendado DD/MM/AAAA. Se ha interpretado como {fecha.strftime('%d/%m/%Y')}.")
            return fecha
        except ValueError:
            continue
    return None

def solicitar_fechas_validas():
    while True:
        nueva_entrada = input("Bot Hotel: Indique las fechas de entrada y salida en formato DD/MM/AAAA: ")
        posibles = extraer_fechas_flexible(nueva_entrada)
        if len(posibles) < 2:
            print("Bot Hotel: No he detectado dos fechas. Inténtelo de nuevo.")
            continue
        fecha_entrada = parsear_fecha(posibles[0])
        fecha_salida = parsear_fecha(posibles[1])
        if not fecha_entrada or not fecha_salida:
            print("Bot Hotel: Formato de fecha no reconocido. Use DD/MM/AAAA.")
            continue
        if fecha_entrada >= fecha_salida:
            print("Bot Hotel: La fecha de entrada debe ser anterior a la de salida.")
            continue
        return fecha_entrada, fecha_salida

## Ejercicio 5: Resumen de reservas con 'mostrar reservas'
Nueva intención listar que imprime un resumen tabulado de todas las reservas activas.

In [6]:
def mostrar_resumen_reservas(reservas):
    if not reservas:
        return "No hay reservas registradas."
    cabecera = f"{'ID':<5}{'Entrada':<14}{'Salida':<14}{'Personas':<10}{'Tipo':<12}{'Total':<10}"
    separador = "-" * len(cabecera)
    filas = [
        f"{r['id']:<5}{r['entrada']:<14}{r['salida']:<14}{r['personas']:<10}{r.get('tipo', '-'):<12}{str(r.get('total', '-')) + '€':<10}"
        for r in reservas
    ]
    return "\n".join([cabecera, separador] + filas)

# Producto final: chatbot integrado
Versión final que integra todos los ejercicios anteriores. Sustituye a la función simular_chat original.

In [8]:
import spacy
import re
from datetime import datetime

nlp = spacy.load("es_core_news_sm")

reservas = []
id_reserva = cargar_reservas(reservas)

def extraer_fecha(texto):
    return re.findall(r'\d{2}/\d{2}/\d{4}', texto)

def validar_fecha(fecha_str):
    try:
        return datetime.strptime(fecha_str, '%d/%m/%Y')
    except ValueError:
        return None

def extraer_personas(texto):
    match = re.search(r'\b(\d+)\s*(personas|persona|huespedes|huéspedes)\b', texto, re.IGNORECASE)
    return int(match.group(1)) if match else None

def detectar_intencion(texto):
    texto = texto.lower()
    if "mostrar reservas" in texto or "ver reservas" in texto or "listar reservas" in texto:
        return "listar"
    if any(p in texto for p in ["piscina", "desayuno", "wifi", "parking", "gimnasio", "spa", "servicio", "servicios", "incluido", "incluye"]):
        return "servicios"
    if any(p in texto for p in ["reservar", "reserva", "habitacion", "habitación"]):
        return "reserva"
    elif "cancelar" in texto:
        return "cancelar"
    elif any(p in texto for p in ["disponibilidad", "precio", "precios", "cuánto", "cuesta", "tarifa", "tarifas"]):
        return "consultar"
    elif any(p in texto for p in ["hola", "buenas", "saludos"]):
        return "saludo"
    elif any(p in texto for p in ["gracias", "adiós", "hasta luego"]):
        return "despedida"
    return "desconocido"

def simular_chat():
    global id_reserva
    print("Bot Hotel: ¡Hola! Bienvenido al servicio de reservas.")
    print("Bot Hotel: Puedo ayudarle a consultar precios, servicios, hacer una reserva, cancelarla o mostrar las existentes.")
    while True:
        entrada = input("Tú: ")
        if entrada.lower() in ["salir", "exit", "adiós"]:
            print("Bot Hotel: ¡Gracias por contactarnos! Que tenga un buen día.")
            break

        intencion = detectar_intencion(entrada)

        if intencion == "saludo":
            print("Bot Hotel: ¡Hola! ¿Desea consultar disponibilidad, servicios, hacer o cancelar una reserva?")

        elif intencion == "servicios":
            print(f"Bot Hotel: {responder_servicios(entrada)}")

        elif intencion == "consultar":
            print(f"Bot Hotel: {mostrar_precios()}\n¿Desea reservar alguna?")

        elif intencion == "listar":
            print(f"Bot Hotel: Resumen de reservas:\n{mostrar_resumen_reservas(reservas)}")

        elif intencion == "cancelar":
            id_cancelar = input("Bot Hotel: Por favor, indique su ID de reserva: ")
            reserva_encontrada = next((r for r in reservas if str(r['id']) == id_cancelar), None)
            if reserva_encontrada:
                reservas.remove(reserva_encontrada)
                guardar_reservas(reservas)
                print(f"Bot Hotel: La reserva {id_cancelar} ha sido cancelada.")
            else:
                print("Bot Hotel: No se ha encontrado ninguna reserva con ese ID.")

        elif intencion == "reserva":
            posibles = extraer_fechas_flexible(entrada)
            fecha_entrada = fecha_salida = None
            if len(posibles) >= 2:
                fecha_entrada = parsear_fecha(posibles[0])
                fecha_salida = parsear_fecha(posibles[1])
                if fecha_entrada and fecha_salida and fecha_entrada >= fecha_salida:
                    fecha_entrada = fecha_salida = None

            if not fecha_entrada or not fecha_salida:
                fecha_entrada, fecha_salida = solicitar_fechas_validas()

            personas = extraer_personas(entrada)
            if not personas:
                while True:
                    nueva_entrada = input("Bot Hotel: ¿Para cuántas personas es la reserva? ")
                    try:
                        personas = int(nueva_entrada)
                        if personas <= 0:
                            raise ValueError
                        break
                    except ValueError:
                        print("Bot Hotel: Número de personas no válido. Introduzca un entero positivo.")

            tipo = extraer_tipo_habitacion(entrada)
            while not tipo:
                nueva_entrada = input(f"Bot Hotel: ¿Qué tipo de habitación desea ({', '.join(TIPOS_HABITACION.keys())})? ")
                tipo = extraer_tipo_habitacion(nueva_entrada)

            noches = (fecha_salida - fecha_entrada).days
            total = noches * TIPOS_HABITACION[tipo]

            reserva = {
                "id": id_reserva,
                "entrada": fecha_entrada.strftime('%d/%m/%Y'),
                "salida": fecha_salida.strftime('%d/%m/%Y'),
                "personas": personas,
                "tipo": tipo,
                "total": total
            }
            reservas.append(reserva)
            guardar_reservas(reservas)
            print(f"Bot Hotel: ¡Reserva confirmada! ID: {id_reserva}. Habitación {tipo} por {noches} noches. Total: {total}€.")
            id_reserva += 1

        elif intencion == "despedida":
            print("Bot Hotel: ¡Gracias por contactarnos! Que tenga un buen día.")
            break
        else:
            print("Bot Hotel: Lo siento, no entendí su mensaje. ¿Puede reformularlo?")

simular_chat()

Bot Hotel: ¡Hola! Bienvenido al servicio de reservas.
Bot Hotel: Puedo ayudarle a consultar precios, servicios, hacer una reserva, cancelarla o mostrar las existentes.
Bot Hotel: No he detectado dos fechas. Inténtelo de nuevo.
Bot Hotel: ¡Reserva confirmada! ID: 1. Habitación doble por 4 noches. Total: 320€.
Bot Hotel: Lo siento, no entendí su mensaje. ¿Puede reformularlo?
Bot Hotel: Lo siento, no entendí su mensaje. ¿Puede reformularlo?
Bot Hotel: Resumen de reservas:
ID   Entrada       Salida        Personas  Tipo        Total     
-----------------------------------------------------------------
1    20/12/2026    24/12/2026    2         doble       320€      
Bot Hotel: Lo siento, no entendí su mensaje. ¿Puede reformularlo?
Bot Hotel: ¡Hola! ¿Desea consultar disponibilidad, servicios, hacer o cancelar una reserva?
Bot Hotel: ¡Gracias por contactarnos! Que tenga un buen día.
